In [80]:
import pandas as pd
import numpy as np
from collections import Counter
import re
import nltk
from nltk.corpus import stopwords
from nltk.util import ngrams

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, classification_report

In [30]:
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\rizwa\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [31]:
df = pd.read_csv(r"../data/customer_support_tickets.csv")
df.head()

,Ticket ID,Customer Name,Customer Email,Customer Age,Customer Gender,Product Purchased,Date of Purchase,Ticket Type,Ticket Subject,Ticket Description,Ticket Status,Resolution,Ticket Priority,Ticket Channel,First Response Time,Time to Resolution,Customer Satisfaction Rating
0,1,Marisa Obrien,carrollallison@example.com,32,Other,GoPro Hero,2021-03-22,Technical issue,Product setup,I'm having an issue with the {product_purchase...,Pending Customer Response,NaN,Critical,Social media,2023-06-01 12:15:36,NaN,NaN
1,2,Jessica Rios,clarkeashley@example.com,42,Female,LG Smart TV,2021-05-22,Technical issue,Peripheral compatibility,I'm having an issue with the {product_purchase...,Pending Customer Response,NaN,Critical,Chat,2023-06-01 16:45:38,NaN,NaN
2,3,Christopher Robbins,gonzalestracy@example.com,48,Other,Dell XPS,2020-07-14,Technical issue,Network problem,I'm facing a problem with my {product_purchase...,Closed,Case maybe show recently my computer follow.,Low,Social media,2023-06-01 11:14:38,2023-06-01 18:05:38,3.0
3,4,Christina Dillon,bradleyolson@example.org,27,Female,Microsoft Office,2020-11-13,Billing inquiry,Account access,I'm having an issue with the {product_purchase...,Closed,Try capital clearly never color toward story.,Low,Social media,2023-06-01 07:29:40,2023-06-01 01:57:40,3.0
4,5,Alexander Carroll,bradleymark@example.com,67,Female,Autodesk AutoCAD,2020-02-04,Billing inquiry,Data loss,I'm having an issue with the {product_purchase...,Closed,West decision evidence bit.,Low,Email,2023-06-01 00:12:42,2023-06-01 19:53:42,1.0


In [32]:
#create new col text that combines subject and description

In [33]:
df["Text"] = (df["Ticket Subject"] + " " + df["Ticket Description"])

In [34]:
df[["Text", "Ticket Subject","Ticket Description"]]

,Text,Ticket Subject,Ticket Description
0,Product setup I'm having an issue with the {pr...,Product setup,I'm having an issue with the {product_purchase...
1,Peripheral compatibility I'm having an issue w...,Peripheral compatibility,I'm having an issue with the {product_purchase...
2,Network problem I'm facing a problem with my {...,Network problem,I'm facing a problem with my {product_purchase...
3,Account access I'm having an issue with the {p...,Account access,I'm having an issue with the {product_purchase...
4,Data loss I'm having an issue with the {produc...,Data loss,I'm having an issue with the {product_purchase...
...,...,...,...
8464,Installation support My {product_purchased} is...,Installation support,My {product_purchased} is making strange noise...
8465,Refund request I'm having an issue with the {p...,Refund request,I'm having an issue with the {product_purchase...
8466,Account access I'm having an issue with the {p...,Account access,I'm having an issue with the {product_purchase...
8467,Payment issue I'm having an issue with the {pr...,Payment issue,I'm having an issue with the {product_purchase...


text cleaning and lowering

In [ ]:
def clean_text(text):
    text = text.lower()
    text = text.replace('_', ' ')
    text = re.sub(r'[^a-zA-Z\s]', '', text) #any character than a letter
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

df['clean_text'] = df['Text'].apply(clean_text)

In [36]:
df[["Text","clean_text"]]

,Text,clean_text
0,Product setup I'm having an issue with the {pr...,product setup im having an issue with the prod...
1,Peripheral compatibility I'm having an issue w...,peripheral compatibility im having an issue wi...
2,Network problem I'm facing a problem with my {...,network problem im facing a problem with my pr...
3,Account access I'm having an issue with the {p...,account access im having an issue with the pro...
4,Data loss I'm having an issue with the {produc...,data loss im having an issue with the product ...
...,...,...
8464,Installation support My {product_purchased} is...,installation support my product purchased is m...
8465,Refund request I'm having an issue with the {p...,refund request im having an issue with the pro...
8466,Account access I'm having an issue with the {p...,account access im having an issue with the pro...
8467,Payment issue I'm having an issue with the {pr...,payment issue im having an issue with the prod...


splitting words

In [37]:
df["Tokens"] = df["clean_text"].apply(lambda x: x.split())
df[["clean_text","Tokens"]]

,clean_text,Tokens
0,product setup im having an issue with the prod...,"[product, setup, im, having, an, issue, with, ..."
1,peripheral compatibility im having an issue wi...,"[peripheral, compatibility, im, having, an, is..."
2,network problem im facing a problem with my pr...,"[network, problem, im, facing, a, problem, wit..."
3,account access im having an issue with the pro...,"[account, access, im, having, an, issue, with,..."
4,data loss im having an issue with the product ...,"[data, loss, im, having, an, issue, with, the,..."
...,...,...
8464,installation support my product purchased is m...,"[installation, support, my, product, purchased..."
8465,refund request im having an issue with the pro...,"[refund, request, im, having, an, issue, with,..."
8466,account access im having an issue with the pro...,"[account, access, im, having, an, issue, with,..."
8467,payment issue im having an issue with the prod...,"[payment, issue, im, having, an, issue, with, ..."


stop words

In [38]:
stop_words = set(stopwords.words('english'))

df['tokens_clean'] = df['Tokens'].apply(lambda tokens: [word for word in tokens if word not in stop_words])

df[["Tokens","tokens_clean"]]

,Tokens,tokens_clean
0,"[product, setup, im, having, an, issue, with, ...","[product, setup, im, issue, product, purchased..."
1,"[peripheral, compatibility, im, having, an, is...","[peripheral, compatibility, im, issue, product..."
2,"[network, problem, im, facing, a, problem, wit...","[network, problem, im, facing, problem, produc..."
3,"[account, access, im, having, an, issue, with,...","[account, access, im, issue, product, purchase..."
4,"[data, loss, im, having, an, issue, with, the,...","[data, loss, im, issue, product, purchased, pl..."
...,...,...
8464,"[installation, support, my, product, purchased...","[installation, support, product, purchased, ma..."
8465,"[refund, request, im, having, an, issue, with,...","[refund, request, im, issue, product, purchase..."
8466,"[account, access, im, having, an, issue, with,...","[account, access, im, issue, product, purchase..."
8467,"[payment, issue, im, having, an, issue, with, ...","[payment, issue, im, issue, product, purchased..."


text statistics

In [39]:
word_counts = Counter(word for tokens in df['tokens_clean'] 
                      for word in tokens)

word_counts.most_common(20)

[('product', 18446),
 ('purchased', 14406),
 ('issue', 13082),
 ('im', 10287),
 ('please', 8808),
 ('assist', 6250),
 ('ive', 6001),
 ('problem', 3385),
 ('data', 2147),
 ('software', 2126),
 ('account', 1971),
 ('support', 1469),
 ('steps', 1390),
 ('error', 1312),
 ('noticed', 1208),
 ('persists', 1178),
 ('help', 1176),
 ('resolve', 1165),
 ('request', 1159),
 ('update', 1155)]

In [40]:
custom_stopwords = {'product','purchased'}

stop_words = set(stopwords.words('english'))
stop_words.update(custom_stopwords)

In [42]:
df['tokens_clean'] = df['Tokens'].apply(lambda tokens: [word for word in tokens if word not in stop_words])

In [44]:
word_counts = Counter(word for tokens in df['tokens_clean']
    for word in tokens)

word_counts.most_common(20)

[('issue', 13082),
 ('im', 10287),
 ('please', 8808),
 ('assist', 6250),
 ('ive', 6001),
 ('problem', 3385),
 ('data', 2147),
 ('software', 2126),
 ('account', 1971),
 ('support', 1469),
 ('steps', 1390),
 ('error', 1312),
 ('noticed', 1208),
 ('persists', 1178),
 ('help', 1176),
 ('resolve', 1165),
 ('request', 1159),
 ('update', 1155),
 ('message', 1129),
 ('would', 1122)]

n gram

In [46]:
bigram_counts = Counter(bigram
    for tokens in df['tokens_clean']
    for bigram in ngrams(tokens, 2))

bigram_counts.most_common(20)

[(('issue', 'please'), 6283),
 (('please', 'assist'), 6237),
 (('im', 'issue'), 6150),
 (('issue', 'im'), 1785),
 (('ive', 'noticed'), 1190),
 (('issue', 'persists'), 1176),
 (('error', 'message'), 971),
 (('ive', 'tried'), 964),
 (('software', 'bug'), 942),
 (('problem', 'im'), 894),
 (('compatibility', 'im'), 862),
 (('im', 'unable'), 853),
 (('data', 'loss'), 852),
 (('im', 'facing'), 836),
 (('battery', 'life'), 831),
 (('request', 'im'), 813),
 (('ive', 'checked'), 747),
 (('resolve', 'problem'), 747),
 (('hardware', 'issue'), 731),
 (('network', 'problem'), 729)]

In [47]:
trigram_counts = Counter(trigram
    for tokens in df['tokens_clean']
    for trigram in ngrams(tokens, 3))

trigram_counts.most_common(20)

[(('issue', 'please', 'assist'), 6053),
 (('im', 'issue', 'please'), 6034),
 (('issue', 'im', 'issue'), 1093),
 (('problem', 'im', 'issue'), 763),
 (('compatibility', 'im', 'issue'), 749),
 (('request', 'im', 'issue'), 708),
 (('issue', 'im', 'facing'), 559),
 (('ive', 'performed', 'factory'), 502),
 (('performed', 'factory', 'reset'), 502),
 (('factory', 'reset', 'hoping'), 502),
 (('reset', 'hoping', 'would'), 502),
 (('hoping', 'would', 'resolve'), 502),
 (('would', 'resolve', 'problem'), 502),
 (('resolve', 'problem', 'didnt'), 502),
 (('problem', 'didnt', 'help'), 502),
 (('im', 'facing', 'intermittent'), 499),
 (('facing', 'intermittent', 'sometimes'), 499),
 (('intermittent', 'sometimes', 'works'), 499),
 (('sometimes', 'works', 'fine'), 499),
 (('works', 'fine', 'times'), 499)]

In [ ]:
#from bigram and trigram some of issues are identified

TF IDF VECTOR

In [49]:
tfidf = TfidfVectorizer(max_features=5000,ngram_range=(1, 2))

X_tfidf = tfidf.fit_transform(df['clean_text'])

print("TF-IDF shape:", X_tfidf.shape)

TF-IDF shape: (8469, 5000)


In [50]:
feature_names = tfidf.get_feature_names_out()

print(feature_names[:50])

['able' 'able to' 'about' 'about it' 'about ive' 'about my' 'about the'
 'about this' 'about to' 'about what' 'about your' 'above' 'accept'
 'accepted' 'access' 'access im' 'access ive' 'access my' 'access the'
 'access there' 'access to' 'accidentally' 'accidentally deleted'
 'account' 'account access' 'account after' 'account and' 'account by'
 'account can' 'account for' 'account has' 'account how' 'account id'
 'account if' 'account im' 'account in' 'account is' 'account it'
 'account ive' 'account login' 'account manager' 'account my'
 'account number' 'account on' 'account please' 'account product'
 'account simply' 'account step' 'account the' 'account there']


In [55]:
row = X_tfidf[0].toarray().flatten()

tfidf_scores = list(zip(feature_names, row))

tfidf_scores = sorted(tfidf_scores,key=lambda x: x[1],reverse=True)

tfidf_scores[:20]

[('address', np.float64(0.2590207312706338)),
 ('we appreciate', np.float64(0.2243652044942326)),
 ('have requested', np.float64(0.22104732230282886)),
 ('address please', np.float64(0.2180793755469136)),
 ('code is', np.float64(0.21294347933720437)),
 ('your billing', np.float64(0.20860114096443788)),
 ('check your', np.float64(0.20483963637157987)),
 ('address ive', np.float64(0.19999871081835724)),
 ('appreciate', np.float64(0.19855380742426088)),
 ('billing', np.float64(0.19461679173215934)),
 ('your email', np.float64(0.1812243722898202)),
 ('requested', np.float64(0.17448540584543168)),
 ('assist your', np.float64(0.16807996126062658)),
 ('your', np.float64(0.15779240510435094)),
 ('email address', np.float64(0.1571397767235941)),
 ('that you', np.float64(0.15157543970282833)),
 ('code', np.float64(0.14294504981221792)),
 ('check', np.float64(0.14015371454580008)),
 ('tried troubleshooting', np.float64(0.13100643968889952)),
 ('website', np.float64(0.12189643832774631))]

In [ ]:
#target

In [58]:
df.columns

Index(['Ticket ID', 'Customer Name', 'Customer Email', 'Customer Age',
       'Customer Gender', 'Product Purchased', 'Date of Purchase',
       'Ticket Type', 'Ticket Subject', 'Ticket Description', 'Ticket Status',
       'Resolution', 'Ticket Priority', 'Ticket Channel',
       'First Response Time', 'Time to Resolution',
       'Customer Satisfaction Rating', 'Text', 'clean_text', 'Tokens',
       'tokens_clean'],
      dtype='str')

In [56]:
print(df['Ticket Type'].value_counts())

Ticket Type
Refund request          1752
Technical issue         1747
Cancellation request    1695
Product inquiry         1641
Billing inquiry         1634
Name: count, dtype: int64


In [75]:
X_text = df['clean_text']
y = df['Ticket Type']

X_train_text, X_test_text, y_train, y_test = train_test_split(X_text,y,test_size=0.2,random_state=42,stratify=y)

In [76]:
tfidf = TfidfVectorizer(max_features=5000,ngram_range=(1, 2))

X_train_tfidf = tfidf.fit_transform(X_train_text)
X_test_tfidf = tfidf.transform(X_test_text)

In [77]:
print("X_train TF-IDF shape:", X_train_tfidf.shape)
print("X_test TF-IDF shape:", X_test_tfidf.shape)

X_train TF-IDF shape: (6775, 5000)
X_test TF-IDF shape: (1694, 5000)


In [78]:
text_model = LogisticRegression(max_iter=1000)

text_model.fit(X_train_tfidf, y_train)

y_pred_lr = text_model.predict(X_test_tfidf)

In [79]:
accuracy_lr = accuracy_score(y_test, y_pred_lr)

print("Logistic Regression Accuracy:", accuracy_lr)
print(classification_report(y_test, y_pred_lr))

Logistic Regression Accuracy: 0.19657615112160567
                      precision    recall  f1-score   support

     Billing inquiry       0.19      0.18      0.19       327
Cancellation request       0.17      0.17      0.17       339
     Product inquiry       0.22      0.22      0.22       328
      Refund request       0.20      0.22      0.21       351
     Technical issue       0.21      0.20      0.20       349

            accuracy                           0.20      1694
           macro avg       0.20      0.20      0.20      1694
        weighted avg       0.20      0.20      0.20      1694



navie bayes

In [81]:
nb_model = MultinomialNB()

nb_model.fit(X_train_tfidf, y_train)

y_pred_nb = nb_model.predict(X_test_tfidf)

In [82]:
accuracy_nb = accuracy_score(y_test, y_pred_nb)

print("Naive Bayes Accuracy:", accuracy_nb)
print(classification_report(y_test, y_pred_nb))

Naive Bayes Accuracy: 0.19480519480519481
                      precision    recall  f1-score   support

     Billing inquiry       0.16      0.12      0.14       327
Cancellation request       0.19      0.19      0.19       339
     Product inquiry       0.20      0.19      0.19       328
      Refund request       0.20      0.22      0.21       351
     Technical issue       0.22      0.24      0.23       349

            accuracy                           0.19      1694
           macro avg       0.19      0.19      0.19      1694
        weighted avg       0.19      0.19      0.19      1694



In [83]:
df.to_csv(r"../data/processed/customer.csv",index=False)